# Phase 1a — TSLA Price Data Collection
**Goal:** Download 4 years of TSLA daily OHLCV data, explore it, and save it clean.

We use `yfinance` which pulls directly from Yahoo Finance — no API key needed.

## Step 1 — Install and import libraries

In [ ]:
# Install yfinance (not pre-installed on Colab)
!pip install yfinance pandas_ta --quiet

In [ ]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## Step 2 — Download TSLA price data

In [ ]:
TICKER = 'TSLA'
START_DATE = '2020-01-01'
END_DATE   = '2024-12-31'

# Download OHLCV data
# OHLCV = Open, High, Low, Close, Volume
df = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True)

# Flatten multi-level columns if present (yfinance sometimes returns them)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f'Downloaded {len(df)} trading days of {TICKER} data')
print(f'Date range: {df.index[0].date()} to {df.index[-1].date()}')
print(f'\nShape: {df.shape}')
df.head(10)

**What you see above:**
- `Open` — price when market opened that day
- `High / Low` — highest and lowest price during the day
- `Close` — price when market closed ← this is what we'll predict
- `Volume` — number of shares traded

## Step 3 — Check data quality

In [ ]:
print('=== Missing values ===')
print(df.isnull().sum())

print('\n=== Basic statistics ===')
print(df.describe().round(2))

print('\n=== Data types ===')
print(df.dtypes)

## Step 4 — Add technical indicators as features

In [ ]:
# --- Daily return ---
# How much did the price change today vs yesterday? (as a %)
# This is more useful than raw price for the model
df['daily_return'] = df['Close'].pct_change()

# --- RSI (Relative Strength Index) ---
# A momentum indicator: above 70 = overbought, below 30 = oversold
# 14-day is the standard window
df['RSI'] = ta.rsi(df['Close'], length=14)

# --- MACD (Moving Average Convergence Divergence) ---
# Shows trend direction and momentum
# Returns 3 columns: MACD line, histogram, signal line
macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
df['MACD']        = macd['MACD_12_26_9']
df['MACD_signal'] = macd['MACDs_12_26_9']
df['MACD_hist']   = macd['MACDh_12_26_9']

# --- Bollinger Bands ---
# Upper/lower bands show if price is unusually high or low
bbands = ta.bbands(df['Close'], length=20, std=2)
df['BB_upper'] = bbands['BBU_20_2.0']
df['BB_lower'] = bbands['BBL_20_2.0']
df['BB_mid']   = bbands['BBM_20_2.0']

# --- Volume change ratio ---
# Is today's trading volume higher or lower than yesterday?
df['volume_change'] = df['Volume'].pct_change()

# --- Moving averages ---
df['MA_20']  = df['Close'].rolling(window=20).mean()
df['MA_50']  = df['Close'].rolling(window=50).mean()

print('Features added!')
print(f'Columns now: {list(df.columns)}')
print(f'\nShape after adding features: {df.shape}')
df.tail(5)

## Step 5 — Drop rows with NaN (from rolling window calculations)

In [ ]:
# The first ~50 rows will have NaN because rolling windows need history
# We drop these cleanly
rows_before = len(df)
df.dropna(inplace=True)
rows_after = len(df)

print(f'Dropped {rows_before - rows_after} rows with NaN values')
print(f'Clean dataset: {rows_after} trading days remaining')
print(f'\nMissing values after clean: {df.isnull().sum().sum()} (should be 0)')

## Step 6 — Visualise TSLA price + key indicators

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
fig.suptitle('TSLA — Price & Technical Indicators (2020–2024)', fontsize=14, y=1.01)

# --- Plot 1: Price with Bollinger Bands and MAs ---
ax1 = axes[0]
ax1.plot(df.index, df['Close'],     color='#1f77b4', linewidth=1.2, label='Close price')
ax1.plot(df.index, df['MA_20'],     color='orange',  linewidth=1,   linestyle='--', label='MA 20')
ax1.plot(df.index, df['MA_50'],     color='green',   linewidth=1,   linestyle='--', label='MA 50')
ax1.fill_between(df.index, df['BB_upper'], df['BB_lower'], alpha=0.1, color='gray', label='Bollinger Bands')
ax1.set_ylabel('Price (USD)')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)

# --- Plot 2: RSI ---
ax2 = axes[1]
ax2.plot(df.index, df['RSI'], color='purple', linewidth=1)
ax2.axhline(70, color='red',   linestyle='--', alpha=0.7, label='Overbought (70)')
ax2.axhline(30, color='green', linestyle='--', alpha=0.7, label='Oversold (30)')
ax2.set_ylabel('RSI')
ax2.set_ylim(0, 100)
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3)

# --- Plot 3: MACD ---
ax3 = axes[2]
ax3.plot(df.index, df['MACD'],        color='blue', linewidth=1,   label='MACD')
ax3.plot(df.index, df['MACD_signal'], color='red',  linewidth=1,   label='Signal')
ax3.bar(df.index,  df['MACD_hist'],   color='gray', alpha=0.4,     label='Histogram')
ax3.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax3.set_ylabel('MACD')
ax3.set_xlabel('Date')
ax3.legend(loc='upper left', fontsize=8)
ax3.grid(True, alpha=0.3)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('tsla_price_indicators.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved!')

## Step 7 — Save clean price data to CSV

In [ ]:
# Reset index so 'Date' becomes a regular column (easier to merge later)
df.index.name = 'Date'
df.reset_index(inplace=True)

# Convert Date to string format YYYY-MM-DD
# This is important for merging with news data later
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')

# Save
df.to_csv('tsla_price_data.csv', index=False)

print('Saved: tsla_price_data.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print('\nFirst 3 rows:')
df.head(3)

## Summary

You now have `tsla_price_data.csv` with:
- **~980 trading days** of TSLA data (2020–2024)
- **Columns:** Date, Open, High, Low, Close, Volume, daily_return, RSI, MACD, MACD_signal, MACD_hist, BB_upper, BB_lower, BB_mid, volume_change, MA_20, MA_50

**Next:** Phase 1b will collect TSLA news headlines for the same date range using RSS feeds.